<a href="https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

For my baseline, I decided to give a higher action score to pages with a low CTR and a high average search position (meaning they rank lower in search results). These pages are getting some visibility, but they are not attracting as many clicks as they could, so they may be good candidates for content optimization.

The score increases when:
- The page has a lower CTR than most other pages.
- The page has a higher average search position (lower ranking).

Pages with the highest scores are placed at the top of the queue and marked for review.

## Reason Codes

- `LOW_CTR` – The page is getting impressions but not enough clicks, which may mean the title, meta description, or content could be improved.
- `POOR_POSITION` – The page is ranking lower in search results, so improving the content may help increase its visibility.

## Action Label

`OPTIMIZE_CONTENT`

In [ ]:
!git clone https://github.com/farankhandev/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 163 (delta 68), reused 103 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.85 MiB | 14.05 MiB/s, done.
Resolving deltas: 100% (68/68), done.


In [ ]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [ ]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Only keep pages with at least 10 impressions
df = df[df["impressions_90d"] >= 10]
ctr_score = 1 - (
    (df["ctr"] - df["ctr"].min()) /
    (df["ctr"].max() - df["ctr"].min())
)

# Poor average position -> higher score
position_score = (
    (df["avg_position"] - df["avg_position"].min()) /
    (df["avg_position"].max() - df["avg_position"].min())
)

# Final score
df["action_score"] = (
    ctr_score * 50 +
    position_score * 50
)




def get_reason(row):
    reasons = []

    if row["ctr"] < df["ctr"].median():
        reasons.append("LOW_CTR")

    if row["avg_position"] > df["avg_position"].median():
        reasons.append("POOR_POSITION")

    return "_".join(reasons) if reasons else "NO_SIGNAL"

df["reason_code"] = df.apply(get_reason, axis=1)

# Action label
df["action"] = "OPTIMIZE_CONTENT"

# ----------------------------
# Rank pages
# ----------------------------

baseline_queue = (
    df.sort_values("action_score", ascending=False)
      .reset_index(drop=True)
)

baseline_queue["rank"] = baseline_queue.index + 1

# Keep useful columns
baseline_queue = baseline_queue[
    [
        "rank",
        "content_id",
        "action_score",
        "reason_code",
        "action",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
]

# ----------------------------
# Save CSV
# ----------------------------

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

baseline_queue.head(10)

CSV saved successfully!


,rank,content_id,action_score,reason_code,action,ctr,avg_position,impressions_90d
0,1,content_7f5783f90bac,100.000000,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,98.6,35
1,2,content_3e262fa8b265,95.283976,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,89.3,12
2,3,content_6b1f3e77da4d,95.182556,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,89.1,75
3,4,content_798197311609,95.081136,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,88.9,1003
4,5,content_933296f93aa5,95.030426,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,88.8,15
5,6,content_3f712db7a02b,94.726166,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,88.2,11
6,7,content_01d9406c6304,94.472617,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,87.7,13
7,8,content_2577dd515a24,94.320487,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,87.4,51
8,9,content_b33dbf37def4,93.762677,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,86.3,10
9,10,content_7b203aad47ab,93.711968,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.0,86.2,35


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
baseline_queue.head(20)

,rank,content_id,action_score,reason_code,action,ctr,avg_position,impressions_90d
0,1,content_7f5783f90bac,100.000000,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,98.6,35
1,2,content_3e262fa8b265,95.283976,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,89.3,12
2,3,content_6b1f3e77da4d,95.182556,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,89.1,75
3,4,content_798197311609,95.081136,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,88.9,1003
4,5,content_933296f93aa5,95.030426,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,88.8,15
5,6,content_3f712db7a02b,94.726166,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,88.2,11
6,7,content_01d9406c6304,94.472617,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,87.7,13
7,8,content_2577dd515a24,94.320487,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,87.4,51
8,9,content_b33dbf37def4,93.762677,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,86.3,10
9,10,content_7b203aad47ab,93.711968,LOW_CTR_POOR_POSITION,OPTIMIZE_CONTENT,0.00,86.2,35


## Top-20 Review

Most of the pages at the top of the list have both a low CTR and a poor average search position. Based on my rule, these pages are the highest priority for content optimization.

| Rank | Action | Reason Code | Confidence Note | What would make it wrong? |
|------|--------|-------------|-----------------|---------------------------|
| 1–20 | OPTIMIZE_CONTENT | LOW_CTR_POOR_POSITION | Medium | The page may target very competitive keywords or new content that has not had enough time to perform. It could also have low impressions, making CTR less reliable. |

In [ ]:
top20 = baseline_queue.head(20).copy()

top20["confidence"] = "Medium"
top20["what_would_make_it_wrong"] = (
    "The page may target competitive keywords or may not have enough data yet."
)

top20[
    [
        "rank",
        "action",
        "reason_code",
        "confidence",
        "what_would_make_it_wrong"
    ]
]

,rank,action,reason_code,confidence,what_would_make_it_wrong
0,1,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
1,2,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
2,3,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
3,4,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
4,5,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
5,6,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
6,7,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
7,8,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
8,9,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...
9,10,OPTIMIZE_CONTENT,LOW_CTR_POOR_POSITION,Medium,The page may target competitive keywords or ma...


## 4. Weak picks + leakage check

## Weak Picks

Some pages have low CTR but also very few impressions. Because there is limited data, these pages may not actually need optimization. I would review them manually before taking action.

## Leakage Check

My rule only uses CTR and average search position. I did not use trend_direction, trend_pct, or any future information, so the baseline does not rely on label leakage.

## Self-check

- [x] Every section is completed.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, URLs, or private information are included.
- [x] My conclusions are based on observed data and are intended for decision support.